In [1]:
!nvidia-smi

Sun Feb 15 15:11:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              1W /   50W |       8MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

def main():
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda version (torch):", torch.version.cuda)
        print("gpu:", torch.cuda.get_device_name(0))
        x = torch.randn(2048, 2048, device="cuda")
        y = x @ x
        torch.cuda.synchronize()
        print("GPU matmul OK. y mean:", y.mean().item())
    else:
        print("No CUDA detected by PyTorch. (If you expected GPU, install a CUDA-enabled torch build.)")

if __name__ == "__main__":
    main()

torch: 2.7.1+cu118
cuda available: True
cuda version (torch): 11.8
gpu: NVIDIA GeForce GTX 1650
GPU matmul OK. y mean: 0.030885891988873482


In [3]:
import json
from pathlib import Path
import torch
import gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,  AutoModelForCausalLM,  BitsAndBytesConfig

In [4]:
def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [5]:
# SETTINGS 
DOCS_DIR = Path("docs")      # 52 .txt files are here
OUT_DIR  = Path("description_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
MODELS = {
    "flan_t5_large": {"type": "seq2seq", "id": "google/flan-t5-large"},
}

In [7]:
# GENERATION SETTINGS 
MAX_INPUT_TOKENS_SEQ2SEQ = 1024       # truncation cap for seq2seq
MIN_NEW_TOKENS = 20
MAX_NEW_TOKENS = 120
NUM_BEAMS = 4

In [9]:
def summarize_seq2seq(model, tokenizer, text: str, device: str) -> str:
    input_text = (
    "Write a brief, 2-3 sentence description of this pharaoh. "
    "Use a factual, engaging tone, like a documentary narrator, "
    "and include vivid details about their reign and achievements:\n" + text
)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding = True,
        max_length=MAX_INPUT_TOKENS_SEQ2SEQ,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}  # move tokenizer outputs to gpu
    
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.2,
            min_new_tokens=MIN_NEW_TOKENS,
            max_new_tokens=MAX_NEW_TOKENS,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

In [10]:
def load_model(model_type: str, model_id: str, device: str):

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    m = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    m.to(device).eval()
    return tok, m

In [11]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    if device == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    files = sorted(DOCS_DIR.glob("*.txt"))
    if not files:
        raise SystemExit(f"No .txt files found in: {DOCS_DIR.resolve()}")

    for model_name, cfg in MODELS.items():
        model_id = cfg["id"]
        model_type = cfg["type"]

        print("\n==============================")
        print(f"Model: {model_name}")
        print(f"HF id: {model_id}")
        print(f"Type : {model_type}")
        print("==============================")

        model_out_dir = OUT_DIR / model_name
        model_out_dir.mkdir(parents=True, exist_ok=True)

        clear_mem()
        tokenizer, model = load_model(model_type, model_id, device)

        for p in files:
            text = p.read_text(encoding="utf-8", errors="ignore").strip()
            print(f"[Text Size] {p.name} -> {len(text.split())} words")
            summary = summarize_seq2seq(model, tokenizer, text, device)
        

            print(f"[Summary Size] {p.name} -> {len(summary.split())} words")

            out_file = model_out_dir / p.name
            out_file.write_text(summary, encoding="utf-8")
            print(f"[Saved] {out_file}")

        # Free memory before next model
        del model
        torch.cuda.empty_cache()
        clear_mem()

In [12]:
if __name__ == "__main__":
    main()

Device: cuda
GPU: NVIDIA GeForce GTX 1650

Model: flan_t5_large
HF id: google/flan-t5-large
Type : seq2seq
[Text Size] Ahmose I.txt -> 1078 words
[Summary Size] Ahmose I.txt -> 47 words
[Saved] description_outputs\flan_t5_large\Ahmose I.txt
[Text Size] Ahmose II.txt -> 987 words
[Summary Size] Ahmose II.txt -> 31 words
[Saved] description_outputs\flan_t5_large\Ahmose II.txt
[Text Size] Alexander The Great.txt -> 3468 words
[Summary Size] Alexander The Great.txt -> 91 words
[Saved] description_outputs\flan_t5_large\Alexander The Great.txt
[Text Size] Amenemhet I.txt -> 1363 words
[Summary Size] Amenemhet I.txt -> 64 words
[Saved] description_outputs\flan_t5_large\Amenemhet I.txt
[Text Size] Amenemhet III.txt -> 1141 words
[Summary Size] Amenemhet III.txt -> 67 words
[Saved] description_outputs\flan_t5_large\Amenemhet III.txt
[Text Size] Amenhotep II.txt -> 1401 words
[Summary Size] Amenhotep II.txt -> 21 words
[Saved] description_outputs\flan_t5_large\Amenhotep II.txt
[Text Size] Amenho